In [31]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [33]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# print("Python:", sys.executable)

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PySpark_Test")
    .master("local[*]")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)


In [5]:
df = spark.read.format('json').load(r'C:\Users\sagar\OneDrive\Desktop\Data Engineering\Resource for DataVidya Cource\PySpark\json data\2015-summary.json')

In [6]:
df.show()

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|       United States|            Romania|   15|
|       United States|            Croatia|    1|
|       United States|            Ireland|  344|
|               Egypt|      United States|   15|
|       United States|              India|   62|
|       United States|          Singapore|    1|
|       United States|            Grenada|   62|
|          Costa Rica|      United States|  588|
|             Senegal|      United States|   40|
|             Moldova|      United States|    1|
|       United States|       Sint Maarten|  325|
|       United States|   Marshall Islands|   39|
|              Guyana|      United States|   64|
|               Malta|      United States|    1|
|            Anguilla|      United States|   41|
|             Bolivia|      United States|   30|
|       United States|           Paraguay|    6|
|             Algeri

In [11]:
df.show(5,vertical= True)

-RECORD 0----------------------------
 DEST_COUNTRY_NAME   | United States 
 ORIGIN_COUNTRY_NAME | Romania       
 count               | 15            
-RECORD 1----------------------------
 DEST_COUNTRY_NAME   | United States 
 ORIGIN_COUNTRY_NAME | Croatia       
 count               | 1             
-RECORD 2----------------------------
 DEST_COUNTRY_NAME   | United States 
 ORIGIN_COUNTRY_NAME | Ireland       
 count               | 344           
-RECORD 3----------------------------
 DEST_COUNTRY_NAME   | Egypt         
 ORIGIN_COUNTRY_NAME | United States 
 count               | 15            
-RECORD 4----------------------------
 DEST_COUNTRY_NAME   | United States 
 ORIGIN_COUNTRY_NAME | India         
 count               | 62            
only showing top 5 rows


In [13]:
df.head(1)

[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count=15)]

In [43]:
# Accessing DataFrame columns
# pyspark.sql.functions import col, column
df.columns

['DEST_COUNTRY_NAME', 'ORIGIN_COUNTRY_NAME', 'count']

In [17]:
df.dtypes

[('DEST_COUNTRY_NAME', 'string'),
 ('ORIGIN_COUNTRY_NAME', 'string'),
 ('count', 'bigint')]

In [19]:
df.printSchema()

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: long (nullable = true)



In [27]:
# Here spark define schema for our df automatically, but in production we have to explicitly define the schema.
# We need to from types StructType, StructField, StringType, LongType (based on data we have)
# we already use "from pyspark.sql.types import *"
# more specifically we can mention: "from pyspark.sql.types import StructType, StructField, StringType, LongType"


In [37]:
my_schema = StructType([
    StructField('DEST_COUNTRY_NAME',StringType(), True),
    StructField('ORIGIN_COUNTRY_NAME',StringType(), True),
    StructField('count',LongType(),False, metadata = {'name': 'Sagar'})
])

# We will redefine our DataFrame

df = spark.read.format('json').schema(my_schema).load(r'C:\Users\sagar\OneDrive\Desktop\Data Engineering\Resource for DataVidya Cource\PySpark\json data\2015-summary.json')

In [39]:
df.show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
|            Egypt|      United States|   15|
|    United States|              India|   62|
+-----------------+-------------------+-----+
only showing top 5 rows


In [ ]:
# You can write select statement in the following ways:

In [83]:
df.select(
    expr('DEST_COUNTRY_NAME as dest'),
    col('DEST_COUNTRY_NAME'),
    column('DEST_COUNTRY_NAME')
).show(2)

+-------------+-----------------+-----------------+
|         dest|DEST_COUNTRY_NAME|DEST_COUNTRY_NAME|
+-------------+-----------------+-----------------+
|United States|    United States|    United States|
|United States|    United States|    United States|
+-------------+-----------------+-----------------+
only showing top 2 rows


In [77]:
df.select(expr('DEST_COUNTRY_NAME as dest')).show(2)

+-------------+
|         dest|
+-------------+
|United States|
|United States|
+-------------+
only showing top 2 rows


In [85]:
df.select(expr('DEST_COUNTRY_NAME').alias('dest')).show(2)

+-------------+
|         dest|
+-------------+
|United States|
|United States|
+-------------+
only showing top 2 rows


In [87]:
# Understanding Transformations-> select, filter, withColumn, when/otherwise

In [89]:

# from pyspark.sql.functions import col, when, lit, upper, concat, year, current_date  ---> we use import *, but this is also imp.

# spark = SparkSession.builder.master("local[*]").getOrCreate() ---> We already created it

data = [
    (1, "alice", "engineering", 95000, "2022-03-15", "active"),
    (2, "bob", "marketing", 72000, "2021-07-01", "active"),
    (3, "carol", "engineering", 110000, "2019-11-20", "active"),
    (4, "dave", "sales", 68000, "2023-01-10", "inactive"),
    (5, "eve", "engineering", 125000, "2018-06-05", "active"),
    (6, "frank", "marketing", 78000, "2022-09-12", "inactive"),
]

df1 = spark.createDataFrame(
    data,
    schema=["emp_id", "name", "department", "salary", "hire_date", "status"]
)


In [94]:
# Understanding select

In [96]:
df1.select('emp_id','name','salary').show()

df1.select(df1.emp_id, df1.name, df1.salary).show()

df1.select(col('emp_id'), col('name'), col('salary')).show()

+------+-----+------+
|emp_id| name|salary|
+------+-----+------+
|     1|alice| 95000|
|     2|  bob| 72000|
|     3|carol|110000|
|     4| dave| 68000|
|     5|  eve|125000|
|     6|frank| 78000|
+------+-----+------+

+------+-----+------+
|emp_id| name|salary|
+------+-----+------+
|     1|alice| 95000|
|     2|  bob| 72000|
|     3|carol|110000|
|     4| dave| 68000|
|     5|  eve|125000|
|     6|frank| 78000|
+------+-----+------+

+------+-----+------+
|emp_id| name|salary|
+------+-----+------+
|     1|alice| 95000|
|     2|  bob| 72000|
|     3|carol|110000|
|     4| dave| 68000|
|     5|  eve|125000|
|     6|frank| 78000|
+------+-----+------+



In [98]:
# Column expression in select
# select is not just for picking columns — it accepts full column expressions

In [102]:
df1.select(
    col('emp_id'),
    upper(col('name')).alias('name'),
    round(col('salary')/12,2).alias('monthly_salary'),
    concat(col('department'), lit('_team')).alias('team')
).show()

+------+-----+--------------+----------------+
|emp_id| name|monthly_salary|            team|
+------+-----+--------------+----------------+
|     1|ALICE|       7916.67|engineering_team|
|     2|  BOB|        6000.0|  marketing_team|
|     3|CAROL|       9166.67|engineering_team|
|     4| DAVE|       5666.67|      sales_team|
|     5|  EVE|      10416.67|engineering_team|
|     6|FRANK|        6500.0|  marketing_team|
+------+-----+--------------+----------------+



In [104]:
# selectExpr — SQL Inside Python

In [112]:
df1.selectExpr(
    'emp_id',
    'upper(name) as name',
    'round(salary/12,2) as monthly_salary',
    "case when monthly_salary > 8000 then 'Senior' else 'Junior' end as grade"
).show()

+------+-----+--------------+------+
|emp_id| name|monthly_salary| grade|
+------+-----+--------------+------+
|     1|ALICE|       7916.67|Junior|
|     2|  BOB|        6000.0|Junior|
|     3|CAROL|       9166.67|Senior|
|     4| DAVE|       5666.67|Junior|
|     5|  EVE|      10416.67|Senior|
|     6|FRANK|        6500.0|Junior|
+------+-----+--------------+------+



In [114]:
# filter/where ---> for selecting rows

In [116]:
df1.filter(col('department') == 'engineering').show()

+------+-----+-----------+------+----------+------+
|emp_id| name| department|salary| hire_date|status|
+------+-----+-----------+------+----------+------+
|     1|alice|engineering| 95000|2022-03-15|active|
|     3|carol|engineering|110000|2019-11-20|active|
|     5|  eve|engineering|125000|2018-06-05|active|
+------+-----+-----------+------+----------+------+



In [126]:
df1.filter( (col('department') == 'engineering') & (col('salary') > 100000) ).show()

df1.filter( (col('department') == 'engineering') | (col('department') == 'marketing') ).show()

df1.filter( ~ (col('status') == 'inactive') ).show()

+------+-----+-----------+------+----------+------+
|emp_id| name| department|salary| hire_date|status|
+------+-----+-----------+------+----------+------+
|     3|carol|engineering|110000|2019-11-20|active|
|     5|  eve|engineering|125000|2018-06-05|active|
+------+-----+-----------+------+----------+------+

+------+-----+-----------+------+----------+--------+
|emp_id| name| department|salary| hire_date|  status|
+------+-----+-----------+------+----------+--------+
|     1|alice|engineering| 95000|2022-03-15|  active|
|     2|  bob|  marketing| 72000|2021-07-01|  active|
|     3|carol|engineering|110000|2019-11-20|  active|
|     5|  eve|engineering|125000|2018-06-05|  active|
|     6|frank|  marketing| 78000|2022-09-12|inactive|
+------+-----+-----------+------+----------+--------+

+------+-----+-----------+------+----------+------+
|emp_id| name| department|salary| hire_date|status|
+------+-----+-----------+------+----------+------+
|     1|alice|engineering| 95000|2022-03-15|

In [132]:
df1.filter(col('name').like('%ve')).show()
df1.filter(col('name').startswith('a')).show()
df1.filter(col('name').contains('ar')).show()

+------+----+-----------+------+----------+--------+
|emp_id|name| department|salary| hire_date|  status|
+------+----+-----------+------+----------+--------+
|     4|dave|      sales| 68000|2023-01-10|inactive|
|     5| eve|engineering|125000|2018-06-05|  active|
+------+----+-----------+------+----------+--------+

+------+-----+-----------+------+----------+------+
|emp_id| name| department|salary| hire_date|status|
+------+-----+-----------+------+----------+------+
|     1|alice|engineering| 95000|2022-03-15|active|
+------+-----+-----------+------+----------+------+

+------+-----+-----------+------+----------+------+
|emp_id| name| department|salary| hire_date|status|
+------+-----+-----------+------+----------+------+
|     3|carol|engineering|110000|2019-11-20|active|
+------+-----+-----------+------+----------+------+



In [134]:
# Understanding withColumn

In [136]:
df1_new = df1.withColumn('bonus', col('salary')*0.1)
df1_new.sh

+------+-----+-----------+------+----------+--------+-------+
|emp_id| name| department|salary| hire_date|  status|  bonus|
+------+-----+-----------+------+----------+--------+-------+
|     1|alice|engineering| 95000|2022-03-15|  active| 9500.0|
|     2|  bob|  marketing| 72000|2021-07-01|  active| 7200.0|
|     3|carol|engineering|110000|2019-11-20|  active|11000.0|
|     4| dave|      sales| 68000|2023-01-10|inactive| 6800.0|
|     5|  eve|engineering|125000|2018-06-05|  active|12500.0|
|     6|frank|  marketing| 78000|2022-09-12|inactive| 7800.0|
+------+-----+-----------+------+----------+--------+-------+



In [138]:
df1_new.withColumn('name', upper(col('name'))).show()

+------+-----+-----------+------+----------+--------+-------+
|emp_id| name| department|salary| hire_date|  status|  bonus|
+------+-----+-----------+------+----------+--------+-------+
|     1|ALICE|engineering| 95000|2022-03-15|  active| 9500.0|
|     2|  BOB|  marketing| 72000|2021-07-01|  active| 7200.0|
|     3|CAROL|engineering|110000|2019-11-20|  active|11000.0|
|     4| DAVE|      sales| 68000|2023-01-10|inactive| 6800.0|
|     5|  EVE|engineering|125000|2018-06-05|  active|12500.0|
|     6|FRANK|  marketing| 78000|2022-09-12|inactive| 7800.0|
+------+-----+-----------+------+----------+--------+-------+



In [144]:
# chaining multiple withColumn

df1_new.withColumn('team', concat(col('department'), lit('_team'))).withColumn('monthly_salary', round(col('salary')/12,2)).withColumn('hire_year', year(col('hire_date'))).show()

+------+-----+-----------+------+----------+--------+-------+----------------+--------------+---------+
|emp_id| name| department|salary| hire_date|  status|  bonus|            team|monthly_salary|hire_year|
+------+-----+-----------+------+----------+--------+-------+----------------+--------------+---------+
|     1|alice|engineering| 95000|2022-03-15|  active| 9500.0|engineering_team|       7916.67|     2022|
|     2|  bob|  marketing| 72000|2021-07-01|  active| 7200.0|  marketing_team|        6000.0|     2021|
|     3|carol|engineering|110000|2019-11-20|  active|11000.0|engineering_team|       9166.67|     2019|
|     4| dave|      sales| 68000|2023-01-10|inactive| 6800.0|      sales_team|       5666.67|     2023|
|     5|  eve|engineering|125000|2018-06-05|  active|12500.0|engineering_team|      10416.67|     2018|
|     6|frank|  marketing| 78000|2022-09-12|inactive| 7800.0|  marketing_team|        6500.0|     2022|
+------+-----+-----------+------+----------+--------+-------+---

In [146]:
# when/otherwise

df1_new.withColumn('grade', when(col('salary') > 100000, 'Senior').when(col('salary') >80000,'mid').otherwise('Junior')).show()

+------+-----+-----------+------+----------+--------+-------+------+
|emp_id| name| department|salary| hire_date|  status|  bonus| grade|
+------+-----+-----------+------+----------+--------+-------+------+
|     1|alice|engineering| 95000|2022-03-15|  active| 9500.0|   mid|
|     2|  bob|  marketing| 72000|2021-07-01|  active| 7200.0|Junior|
|     3|carol|engineering|110000|2019-11-20|  active|11000.0|Senior|
|     4| dave|      sales| 68000|2023-01-10|inactive| 6800.0|Junior|
|     5|  eve|engineering|125000|2018-06-05|  active|12500.0|Senior|
|     6|frank|  marketing| 78000|2022-09-12|inactive| 7800.0|Junior|
+------+-----+-----------+------+----------+--------+-------+------+

